In [1]:
import os
from PIL import Image, ImageEnhance
import numpy as np

# 1. Configuration
INPUT_FOLDER = "./Selected Image for project"
OUTPUT_FOLDER = "./HR_256"

# Create output folder if it doesn't exist
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

# 2. Define the Augmentation Multipliers

def get_9_crops(img, target_size=256):
    """Multiplier 1: Resizes image to 768x768 and slices into a 3x3 grid (9 images)"""
    img = img.resize((target_size * 3, target_size * 3), Image.BICUBIC)
    crops = []
    for i in range(3):
        for j in range(3):
            left = j * target_size
            upper = i * target_size
            right = left + target_size
            lower = upper + target_size
            crops.append(img.crop((left, upper, right, lower)))
    return crops

def get_rotations(img):
    """Multiplier 2: 4 Rotations (360/0, 90, 180, 270)"""
    return [
        img,                             # 360/Original
        img.rotate(90, expand=True),     # 90
        img.rotate(180, expand=True),    # 180
        img.rotate(270, expand=True)     # 270
    ]

def get_flips(img):
    """Multiplier 3: 2 Flips (Original, Horizontal Flip)"""
    return [
        img,                                     # Original
        img.transpose(Image.FLIP_LEFT_RIGHT)     # Horizontal
    ]

def get_brightness(img):
    """Multiplier 4: 6 Brightness Levels (0.3 to 2.0)"""
    enhancer = ImageEnhance.Brightness(img)
    # [0.3 (Very Dark), 0.6, 1.0 (Normal), 1.3, 1.6, 2.0 (Very Bright)]
    factors = [0.3, 0.6, 1.0, 1.3, 1.6, 2.0] 
    return [enhancer.enhance(f) for f in factors]

def get_color_jitter(img):
    """Multiplier 5: 3 Colors (Normal, Warm, Cool)"""
    img = img.convert('RGB')
    r, g, b = img.split()
    
    # Normal
    normal = img
    
    # Warm (Boost Red, Drop Blue)
    r_warm = r.point(lambda i: min(255, int(i * 1.15)))
    b_warm = b.point(lambda i: int(i * 0.85))
    warm = Image.merge('RGB', (r_warm, g, b_warm))
    
    # Cool (Drop Red, Boost Blue)
    r_cool = r.point(lambda i: int(i * 0.85))
    b_cool = b.point(lambda i: min(255, int(i * 1.15)))
    cool = Image.merge('RGB', (r_cool, g, b_cool))
    
    return [normal, warm, cool]

# 3. The Master Pipeline Engine
def process_single_image(image_path, filename):
    try:
        original_img = Image.open(image_path).convert('RGB')
    except Exception as e:
        print(f"Skipping {filename} - Not a valid image.")
        return 0

    base_name = os.path.splitext(filename)[0]
    counter = 1
    
    # Pipeline Execution (The Nested Loop of 1296)
    crops = get_9_crops(original_img)             # 1 -> 9
    for crop in crops:
        rotations = get_rotations(crop)           # 9 -> 36
        for rot in rotations:
            flips = get_flips(rot)                # 36 -> 72
            for flip in flips:
                brights = get_brightness(flip)    # 72 -> 432
                for bright in brights:
                    colors = get_color_jitter(bright) # 432 -> 1296
                    for final_img in colors:
                        
                        # Save the final permutation
                        save_name = f"{base_name}_aug_{counter:04d}.png"
                        save_path = os.path.join(OUTPUT_FOLDER, save_name)
                        final_img.save(save_path, "PNG")
                        counter += 1
                        
    return counter - 1 # Should return 1296

# 4. Execute on the HR_256 Folder
print(f"🚀 Starting Extreme Augmentation on {INPUT_FOLDER}...")

if not os.path.exists(INPUT_FOLDER):
    print(f"🚨 ERROR: Cannot find folder '{INPUT_FOLDER}'. Make sure it is in the same directory.")
else:
    valid_files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    total_files = len(valid_files)
    
    grand_total_generated = 0
    
    for i, filename in enumerate(valid_files):
        img_path = os.path.join(INPUT_FOLDER, filename)
        
        print(f"Processing Image [{i+1}/{total_files}]: {filename}...")
        generated = process_single_image(img_path, filename)
        grand_total_generated += generated
        
    print("--------------------------------------------------")
    print(f"✅ PIPELINE COMPLETE!")
    print(f"Original Images Processed: {total_files}")
    print(f"Total New Images Generated: {grand_total_generated}")
    print(f"Output Directory: {OUTPUT_FOLDER}")

🚀 Starting Extreme Augmentation on ./Selected Image for project...
Processing Image [1/110]: TImage (75).png...
Processing Image [2/110]: TImage (31).png...
Processing Image [3/110]: TImage (103).png...
Processing Image [4/110]: TImage (95).png...
Processing Image [5/110]: TImage (69).png...
Processing Image [6/110]: TImage (46).png...
Processing Image [7/110]: TImage (78).png...
Processing Image [8/110]: TImage (43).png...
Processing Image [9/110]: TImage (44).png...
Processing Image [10/110]: TImage (5).png...
Processing Image [11/110]: TImage (37).png...
Processing Image [12/110]: TImage (47).png...
Processing Image [13/110]: TImage (71).png...
Processing Image [14/110]: TImage (84).png...
Processing Image [15/110]: TImage (97).png...
Processing Image [16/110]: TImage (10).png...
Processing Image [17/110]: TImage (45).png...
Processing Image [18/110]: TImage (1).png...
Processing Image [19/110]: TImage (88).png...
Processing Image [20/110]: TImage (26).png...
Processing Image [21/11

In [2]:
import os

folder = "HR_256"
files = sorted(os.listdir(folder))

for i, filename in enumerate(files):
    ext = os.path.splitext(filename)[1]
    new_name = f"{i:06d}{ext}"
    os.rename(os.path.join(folder, filename), os.path.join(folder, new_name))
    
print("Done!")

Done!


In [1]:
import cv2
from pathlib import Path

PROJECT_ROOT = Path.home() / 'sr_project'
HR_DIR = PROJECT_ROOT / 'HR_256'
LR_DIR = PROJECT_ROOT / 'LR_x4'
LR_DIR.mkdir(parents=True, exist_ok=True)

scale = 4  # 256 → 64

hr_paths = sorted(list(HR_DIR.glob("*.jpg")) + 
                  list(HR_DIR.glob("*.png")) +
                  list(HR_DIR.glob("*.jpeg")))

total = len(hr_paths)
print(f"Found {total} HR images. Starting conversion...")

skipped = 0
for i, p in enumerate(hr_paths):
    out_path = LR_DIR / p.name

    # Skip if already converted (resume-safe)
    if out_path.exists():
        skipped += 1
        continue

    img = cv2.imread(str(p))
    if img is None:
        print(f"  ⚠️  Skipping unreadable file: {p.name}")
        continue

    h, w = img.shape[:2]

    # Downsample ×4 (256→64)
    lr = cv2.resize(img, (w // scale, h // scale), interpolation=cv2.INTER_AREA)

    # Slight blur (realistic degradation)
    lr = cv2.GaussianBlur(lr, (3, 3), 0)

    cv2.imwrite(str(out_path), lr)

    # Progress every 5000 images
    if (i + 1) % 5000 == 0:
        print(f"  ✅ {i+1}/{total} done...")

print(f"\n✅ Conversion complete!")
print(f"   HR images : {total}")
print(f"   LR images : {len(list(LR_DIR.glob('*.jpg'))) + len(list(LR_DIR.glob('*.png')))}")
print(f"   Skipped   : {skipped} (already existed)")

Found 149490 HR images. Starting conversion...
  ✅ 5000/149490 done...
  ✅ 10000/149490 done...
  ✅ 15000/149490 done...
  ✅ 20000/149490 done...
  ✅ 25000/149490 done...
  ✅ 30000/149490 done...
  ✅ 35000/149490 done...
  ✅ 40000/149490 done...
  ✅ 45000/149490 done...
  ✅ 50000/149490 done...
  ✅ 55000/149490 done...
  ✅ 60000/149490 done...
  ✅ 65000/149490 done...
  ✅ 70000/149490 done...
  ✅ 75000/149490 done...
  ✅ 80000/149490 done...
  ✅ 85000/149490 done...
  ✅ 90000/149490 done...
  ✅ 95000/149490 done...
  ✅ 100000/149490 done...
  ✅ 105000/149490 done...
  ✅ 110000/149490 done...
  ✅ 115000/149490 done...
  ✅ 120000/149490 done...
  ✅ 125000/149490 done...
  ✅ 130000/149490 done...
  ✅ 135000/149490 done...
  ✅ 140000/149490 done...
  ✅ 145000/149490 done...

✅ Conversion complete!
   HR images : 149490
   LR images : 149490
   Skipped   : 0 (already existed)


In [1]:
from pathlib import Path
from PIL import Image, ImageFilter

PROJECT_ROOT = Path.home() / 'sr_project'
TEST_DIR     = PROJECT_ROOT / 'Test'
TEST_DIR.mkdir(parents=True, exist_ok=True)

# Collect original images (skip already-generated files) 
all_imgs = sorted([
    p for p in TEST_DIR.iterdir()
    if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp', '.bmp')
    and not p.stem.endswith('_LR')
    and not p.stem.endswith('_ESPCN')
    and not p.stem.endswith('_COMPARE')
])

print(f"Found {len(all_imgs)} image(s) in {TEST_DIR}\n")
assert len(all_imgs) > 0, f"❌ No images found in {TEST_DIR}. Add some images first!"

#  Generate LR for each image 
for img_path in all_imgs:
    print(f"{'='*60}")
    print(f"Processing : {img_path.name}")

    orig         = Image.open(img_path).convert('RGB')
    orig_w, orig_h = orig.size
    print(f"  Original size : {orig_w}×{orig_h}")

    # Step 1 — Crop to multiple of 4 (required for ×4 pipeline)
    hr_w = (orig_w // 4) * 4
    hr_h = (orig_h // 4) * 4
    hr   = orig.resize((hr_w, hr_h), Image.LANCZOS)

    # Step 2 — Downscale ÷4 with bicubic
    lr_w = hr_w // 4
    lr_h = hr_h // 4
    lr   = hr.resize((lr_w, lr_h), Image.BICUBIC)

    # Step 3 — Gaussian blur (same degradation used during training)
    lr   = lr.filter(ImageFilter.GaussianBlur(radius=0.5))

    # Step 4 — Save LR image
    lr_save_path = TEST_DIR / f"{img_path.stem}_LR{img_path.suffix}"
    lr.save(lr_save_path)

    print(f"  LR size       : {lr_w}×{lr_h}")
    print(f"  ✅ Saved       : {lr_save_path.name}")

print(f"\n{'='*60}")
print(f"✅ Part 1 done!  All LR images saved in {TEST_DIR}")
print(f"   Now run  part2_espcn_compare.py  to super-resolve them.")
print(f"{'='*60}")

Found 6 image(s) in /home/rokib/sr_project/Test

Processing : 6529fb046efd17541e77b547bb95f38638f039b3babdf61d.jpg
  Original size : 1920×1080
  LR size       : 480×270
  ✅ Saved       : 6529fb046efd17541e77b547bb95f38638f039b3babdf61d_LR.jpg
Processing : Android-17-3-1200x1200.jpg
  Original size : 1200×1200
  LR size       : 300×300
  ✅ Saved       : Android-17-3-1200x1200_LR.jpg
Processing : How-to-photograph-a-perfume-object-on-a-dark-background.png
  Original size : 1276×1276
  LR size       : 319×319
  ✅ Saved       : How-to-photograph-a-perfume-object-on-a-dark-background_LR.png
Processing : Postal_Service_250_76271.png
  Original size : 1200×1200
  LR size       : 300×300
  ✅ Saved       : Postal_Service_250_76271_LR.png
Processing : closeup-shot-beautiful-butterfly-with-interesting-textures-orange-petaled-flower_181624-7640.png
  Original size : 740×491
  LR size       : 185×122
  ✅ Saved       : closeup-shot-beautiful-butterfly-with-interesting-textures-orange-petaled-flower_